# Data in Hand

## 1. Driving Question

For my final project, I want to use my Spotify listening history and weather data to find out what types of music I listen to most in different seasons. I think I tend to play heavier genres like metal when it’s cold and gloomy, and more pop music when it’s warm and sunny.

I’m using two datasets for this project: my Spotify listening history and daily weather records. Each one gives me part of the picture. Spotify shows what I listened to and when, while the weather data tells me the temperature on those days. I match them up by date.

## 2. The Data

### 2.1 Dataset 1: Spotify Extended Streaming History

**How I collected it**

I asked Spotify for my data through my account and chose the Extended streaming history option. A few hours later, Spotify sent me a download link. The ZIP file had Streaming_History_Audio_2026.json, which is the single JSON file I’ll use for this assignment.

**Privacy**

The file had my IP address for every song I played, which could show where I live. I delete this column right after loading the data, and I’m not uploading the raw JSON file to GitHub since it’s in my .gitignore. Instead, I save a copy without the IP address column as spotify_history_2026_clean.csv and include that in my repository, so the notebook can be run without the raw file.

**Structure**

The file has 9,528 records, each representing a play of a song, podcast episode, or audiobook chapter between January 1 and November 15, 2026. Each record has the same 23 fields, so it loads easily into a table with one row per play. The fields answer four main questions about each play. When did it happen? The ts field gives the time a play ended, stored as text in UTC, so I will convert it to a date and to Eastern time. What was playing? For songs, the fields include the track name, artist, album, and a unique song ID. Podcasts and audiobooks have their own fields, so each row leaves the fields for the other content types empty. How long did I listen? The ms_played field records this as a whole number of milliseconds. How was I listening? Several fields describe this: the device I used, why a play started and ended (for example, the song finished or I pressed skip), and true/false flags for shuffle, skipping, offline mode, or private listening. The file also includes my country (always the US) and my IP address, which I remove for privacy, leaving 22 columns. The file does not have genre information or weather or detailed location, so I have to add those from other sources.

I needed help understanding the JSON file structure, and Claude helped me with that.

In [5]:
# This section, Claude helped me to figure out how to remove my IP address to meet privacy requirements

import json
from pathlib import Path
import pandas as pd

RAW_FILE = Path("Streaming_History_Audio_2026.json")
CLEAN_FILE = Path("spotify_history_2026_clean.csv")

if RAW_FILE.exists():
    with open(RAW_FILE, encoding="utf-8") as f:
        raw = json.load(f)
    df_raw = pd.DataFrame(raw)
    df_raw = df_raw.drop(columns=["ip_addr"])   # remove IP address for privacy
    df_raw.to_csv(CLEAN_FILE, index=False)
else:
    df_raw = pd.read_csv(CLEAN_FILE)

df_raw.head()

,ts,platform,ms_played,conn_country,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2026-01-01T19:36:04Z,ios,0,US,When Did You Get Hot?,Sabrina Carpenter,Man’s Best Friend,spotify:track:0yIUSblMktt9VmCJ133gvY,NaN,NaN,...,NaN,NaN,NaN,trackdone,unexpected-exit-while-paused,False,False,False,1767130846,False
1,2026-01-01T19:38:34Z,ios,145399,US,When Did You Get Hot?,Sabrina Carpenter,Man’s Best Friend,spotify:track:0yIUSblMktt9VmCJ133gvY,NaN,NaN,...,NaN,NaN,NaN,appload,trackdone,False,False,False,1767296164,False
2,2026-01-01T19:42:03Z,ios,208211,US,Never Getting Laid,Sabrina Carpenter,Man’s Best Friend,spotify:track:5wIQSTbtONKXUjxMeXDyGL,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1767296314,False
3,2026-01-01T19:45:56Z,ios,232379,US,Such A Funny Way,Sabrina Carpenter,Man’s Best Friend,spotify:track:5INY1sXYVmFCN5I6cbwdmk,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1767296523,False
4,2026-01-01T19:48:36Z,ios,160184,US,Tears,Sabrina Carpenter,Man’s Best Friend,spotify:track:5P9vEHtAeviz31Flyv6enq,NaN,NaN,...,NaN,NaN,NaN,trackdone,trackdone,True,False,False,1767296756,False


In [6]:
print(f"Rows: {len(df_raw):,}   Columns: {df_raw.shape[1]}")
print("Date range:", df_raw["ts"].min(), "to", df_raw["ts"].max())
print("\nPlays by platform:\n", df_raw["platform"].value_counts())
print("\nPodcast plays:", df_raw["episode_name"].notna().sum())
print("Audiobook plays:", df_raw["audiobook_title"].notna().sum())
print("Rows with no song:", df_raw["spotify_track_uri"].isna().sum())
print("\nShuffle on:", f"{df_raw['shuffle'].mean():.0%}")
print("Skipped:", f"{df_raw['skipped'].mean():.0%}")
df_raw["ms_played"].describe()

Rows: 9,528   Columns: 22
Date range: 2026-01-01T19:36:04Z to 2026-11-15T18:46:44Z

Plays by platform:
 platform
ios        8793
windows     735
Name: count, dtype: int64

Podcast plays: 25
Audiobook plays: 190
Rows with no song: 216

Shuffle on: 75%
Skipped: 45%


count    9.528000e+03
mean     1.245969e+05
std      1.618664e+05
min      0.000000e+00
25%      6.244500e+03
50%      1.137180e+05
75%      2.036098e+05
max      2.799301e+06
Name: ms_played, dtype: float64

**Data types**

The table below shows the type pandas assigned to each column and how many values are filled in. Text fields are stored as object, ms_played and offline_timestamp are whole numbers, and the shuffle, skip, offline, and private-listening flags are true/false values. The podcast and audiobook columns have only a few hundred filled values because most rows are songs.

In [7]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 9528 entries, 0 to 9527
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   ts                                 9528 non-null   str  
 1   platform                           9528 non-null   str  
 2   ms_played                          9528 non-null   int64
 3   conn_country                       9528 non-null   str  
 4   master_metadata_track_name         9312 non-null   str  
 5   master_metadata_album_artist_name  9312 non-null   str  
 6   master_metadata_album_album_name   9312 non-null   str  
 7   spotify_track_uri                  9312 non-null   str  
 8   episode_name                       25 non-null     str  
 9   episode_show_name                  25 non-null     str  
 10  spotify_episode_uri                25 non-null     str  
 11  audiobook_title                    190 non-null    str  
 12  audiobook_uri                  

In [8]:
music = df_raw[df_raw["spotify_track_uri"].notna()]

# Dates in Eastern time so late-night plays land on the right day
dates_et = pd.to_datetime(df_raw["ts"], utc=True).dt.tz_convert("America/New_York").dt.date
days_in_range = (max(dates_et) - min(dates_et)).days + 1

print(f"Total listening: {df_raw['ms_played'].sum() / 3.6e6:,.0f} hours")
print(f"Average play: {df_raw['ms_played'].mean() / 60000:.1f} minutes")
print(f"25th percentile play: {df_raw['ms_played'].quantile(0.25) / 1000:.1f} seconds")
print(f"Unique songs: {music['spotify_track_uri'].nunique():,}")
print(f"Unique artists: {music['master_metadata_album_artist_name'].nunique():,}")
print(f"Days with listening: {dates_et.nunique()} of {days_in_range}")

short = music["ms_played"] < 30000
print(f"\nMusic plays under 30 seconds: {short.mean():.0%}")
print(f"Share of music listening time kept after 30-second filter: "
      f"{music.loc[~short, 'ms_played'].sum() / music['ms_played'].sum():.0%}")

Total listening: 330 hours
Average play: 2.1 minutes
25th percentile play: 6.2 seconds
Unique songs: 1,680
Unique artists: 585
Days with listening: 282 of 319

Music plays under 30 seconds: 38%
Share of music listening time kept after 30-second filter: 98%


**Summary of the Spotify data**

In total, the file has 9,528 plays, adding up to about 330 hours of listening. Almost all of these are music. Only 216 rows have no song: 25 are podcast episodes, 190 are audiobook chapters, and 1 is unidentified. I remove these since they have no music genre. The music data is more varied, with 1,680 different songs by 585 different artists. Most of my listening is on my phone, which makes up 8,793 plays (92%), while my laptop has the other 735 (8%). The data covers 282 out of 319 days in the date range, so only a few days are missing. The data also shows my listening habits. Shuffle was on for 75% of plays, and 45% of plays were skipped. The average play lasted about 2.1 minutes, but a quarter of all plays lasted about 6 seconds or less.

**Data quality issues and how I will handle them**

I found four issues to address before analysis. First, many plays are very short because I shuffle and skip a lot, and just a few seconds of a song doesn’t really count as listening. Spotify only counts a play as a "stream" after 30 seconds, so I use the same rule. This removes 38% of music plays but keeps most of my actual listening time. Second, timestamps are recorded in UTC, so a song I played at 9 pm in Ohio shows up as the next day. I convert everything to Eastern time so each play matches the right day's weather. Third, the missing values in the file are not errors. Music rows are empty in the podcast and audiobook fields, and the reverse is also true. Once I keep only music, the song and artist fields have no missing values. Finally, there is no genre information, so I assign genres myself by artist, which is a simplification for artists who mix genres.

### 2.2 Dataset 2: Daily Weather (Open-Meteo)

**How I collected it**

To get weather data, I used the Open-Meteo Historical Weather API (archive-api.open-meteo.com/v1/archive). It’s free, doesn’t need an API key, and gives daily historical weather for any location. I pulled data for Kent, Ohio (41.15° N, 81.36° W), where I live during the semester, for the same dates as my Spotify data. I chose temperatures in Fahrenheit and set the timezone to Eastern Time so the days match my Spotify timestamps. I saved the results as weather_kent_2026.csv, so I can repeat the analysis later even if the API changes.

**Structure**

The API gives one record for each day, which I put into a table with one row per date. Each row has the date and four numbers: the day’s average temperature, high and low temperatures (all in degrees F), and total precipitation in inches. I mainly use the average daily temperature to test my question, but I keep the high, low, and precipitation in case I want to do more analysis.

One limitation is that I’m assuming I was in Kent every day. If I was traveling or home on break, the weather data won’t match where I actually was.

In [9]:
import requests

params = {
    "latitude": 41.1537, "longitude": -81.3579,
    "start_date": "2026-01-01", "end_date": "2026-11-15",
    "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum",
    "temperature_unit": "fahrenheit",
    "precipitation_unit": "inch",
    "timezone": "America/New_York",
}
resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=30)
resp.raise_for_status()

weather = pd.DataFrame(resp.json()["daily"]).rename(columns={
    "time": "date", "temperature_2m_mean": "temp_mean",
    "temperature_2m_max": "temp_max", "temperature_2m_min": "temp_min",
    "precipitation_sum": "precip_in",
})
weather.to_csv("weather_kent_2026.csv", index=False)
weather.describe().round(1)

,temp_mean,temp_max,temp_min,precip_in
count,319.0,319.0,319.0,319.0
mean,54.1,63.0,45.2,0.1
std,19.1,20.1,19.0,0.2
min,1.1,8.5,-9.6,0.0
25%,42.9,52.0,34.6,0.0
50%,58.9,68.8,49.5,0.0
75%,68.8,77.6,60.2,0.2
max,85.4,94.8,75.6,1.0


In [10]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 319 entries, 0 to 318
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       319 non-null    str    
 1   temp_mean  319 non-null    float64
 2   temp_max   319 non-null    float64
 3   temp_min   319 non-null    float64
 4   precip_in  319 non-null    float64
dtypes: float64(4), str(1)
memory usage: 15.7 KB


### 2.3 Data in Hand: Summary

I have both datasets I need for this project. The Spotify file has detailed listening data, including times and artists for almost every day of 2026 so far. The weather API provides the temperature for each of those days. After I convert the Spotify timestamps to Eastern time, I can match the two datasets by date. The only thing missing is genre, which I will add by labeling artists in the next stage.